# 잡코리아 인재검색 & 입사제안
셀1 → 셀2 → 셀3 → (로그인 필요시 셀4) → 셀5 → 셀6 → 셀7 → 셀8 → 셀9

In [55]:
# ── [셀1] 라이브러리 & 설정 ────────────────────────────────
import os, time, random
from pathlib import Path
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

_cwd = Path(os.getcwd())
for _p in [_cwd / '.env', _cwd / '10.rpa/70.webs/recruit/.env']:
    if _p.exists():
        load_dotenv(_p, override=True)
        print(f'.env: {_p}')
        break

JOBKOREA_ID = os.getenv('JOBKOREA_ID', '')
JOBKOREA_PW = os.getenv('JOBKOREA_PW', '')
PROFILE_DIR = str(Path.home() / '.jobkorea_profile')

assert JOBKOREA_ID, '❌ JOBKOREA_ID 없음 — .env 확인'
assert JOBKOREA_PW, '❌ JOBKOREA_PW 없음 — .env 확인'
print(f'✅ ID: {JOBKOREA_ID[:3]}*** | 프로필: {PROFILE_DIR}')

# ── 랜덤 대기 헬퍼 ──────────────────────────────────────
def rn(lo=0.5, hi=2.0): return random.uniform(lo, hi)   # 검색·설정 단계 (셀2~6)
def rp(lo=1.0, hi=5.0): return random.uniform(lo, hi)   # 포지션 제안 단계 (셀10)
print('✅ 랜덤 대기: rn(검색/설정, 0.5~2.0s)  rp(제안 발송, 1.0~5.0s)')

.env: d:\drive_files\10.worksfree\10.rpa\70.webs\recruit\.env
✅ ID: nam*** | 프로필: C:\Users\USER\.jobkorea_profile
✅ 랜덤 대기: rn(검색/설정, 0.5~2.0s)  rp(제안 발송, 1.0~5.0s)


In [56]:
# ── [셀2] Chrome 브라우저 시작 ─────────────────────────
import json

try:
    driver.quit()
    time.sleep(rn(1.5, 3.0))
    print('이전 브라우저 종료')
except:
    pass

# ── 프로필 잠금 파일 정리 ──
_lock = Path(PROFILE_DIR) / 'SingletonLock'
if _lock.exists():
    _lock.unlink()
    print('⚠️  SingletonLock 삭제')
    time.sleep(rn())

# ── "페이지 복원" 팝업 차단 ──────────────────────────────
# Chrome Preferences 파일의 종료 상태를 Normal로 수정
# (이 파일에 Crashed 상태가 남아있으면 Chrome이 복원 팝업을 띄움)
_prefs_path = Path(PROFILE_DIR) / 'Default' / 'Preferences'
if _prefs_path.exists():
    try:
        _prefs = json.loads(_prefs_path.read_text(encoding='utf-8'))
        _prefs.setdefault('profile', {}).update({
            'exit_type': 'Normal',
            'exited_cleanly': True,
        })
        _prefs_path.write_text(json.dumps(_prefs), encoding='utf-8')
        print('✅ Preferences 복원 상태 초기화 (복원 팝업 차단)')
    except Exception as e:
        print(f'⚠️  Preferences 수정 실패: {e}')

options = Options()
options.page_load_strategy = 'eager'
options.add_argument(f'--user-data-dir={PROFILE_DIR}')
options.add_argument('--profile-directory=Default')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--start-maximized')
options.add_argument('--no-first-run')
options.add_argument('--no-default-browser-check')
options.add_experimental_option('prefs', {
    'profile.exit_type': 'Normal',
    'profile.exited_cleanly': True,
})

print('🚀 Chrome 시작 중...')
service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
except Exception as e:
    print(f'⚠️  1차 실패 → 재시도')
    if _lock.exists(): _lock.unlink()
    time.sleep(rn(2.0, 4.0))
    driver = webdriver.Chrome(service=service, options=options)

driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument',
    {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'})
driver.maximize_window()
print('✅ 브라우저 시작 완료')

이전 브라우저 종료
✅ Preferences 복원 상태 초기화 (복원 팝업 차단)
🚀 Chrome 시작 중...
✅ 브라우저 시작 완료


In [57]:
# ── [셀3] 인재검색 URL 접속 → 로그인 상태 확인 ──────────
import time
from selenium.common.exceptions import TimeoutException as SeleniumTimeout

TALENT_URL      = 'https://www.jobkorea.co.kr/corp/person/find'
USER_NAME_XPATH = '//*[@id="wrap"]/div[2]/div[1]/div[1]/ul/li[1]/a'  # 로그인 시 남산HR

# 페이지 로드 타임아웃 설정: 10초 안에 로드 안 되면 강제 중단
driver.set_page_load_timeout(10)

try:
    driver.get(TALENT_URL)
    # 여기까지 왔으면 10초 내 eager 로드 완료
    time.sleep(rn(1.5, 2.5))
except SeleniumTimeout:
    # 10초 초과 → window.stop()으로 강제 중단 후 계속
    print('⚠️  페이지 로드 10초 초과 → 강제 중단 후 진행')
    try: driver.execute_script('window.stop()')
    except: pass
    time.sleep(rn(1.0, 2.0))
except Exception as e:
    print(f'⚠️  페이지 이동 실패: {type(e).__name__}')
    try: driver.execute_script('window.stop()')
    except: pass

# 로드 여부와 무관하게 항상 한번 더 중단 (잔여 XHR 차단)
try: driver.execute_script('window.stop()')
except: pass

try:
    driver.switch_to.alert.dismiss()
except: pass

for sel in ['button[aria-label="닫기"]', '.popup-close', '.btn-close', '.modal-close']:
    for btn in driver.find_elements(By.CSS_SELECTOR, sel):
        try:
            if btn.is_displayed():
                btn.click()
                time.sleep(rn())
        except: pass

print(f'   URL  : {driver.current_url}')
print(f'   Title: {driver.title}')

logged_in = False
try:
    el = driver.find_element(By.XPATH, USER_NAME_XPATH)
    txt = el.text.strip()
    print(f'   li[1]/a 텍스트: "{txt}"')
    if txt == '남산HR':
        logged_in = True
        print('✅ 로그인 확인 (남산HR)')
except Exception as e:
    print(f'   li[1]/a 없음: {e}')

if logged_in:
    print('✅ 로그인 상태 → 셀5로 진행')
else:
    print('🔑 로그인 필요 → [셀4] 실행')

⚠️  페이지 로드 10초 초과 → 강제 중단 후 진행
   URL  : https://www.jobkorea.co.kr/corp/person/find
   Title: 인재검색 - 직무 경험이 풍부한 우수 인재 | 잡코리아
   li[1]/a 텍스트: "회원가입"
🔑 로그인 필요 → [셀4] 실행


In [58]:
# ── [셀4-1] 로그인 (필요한 경우만 실행) ──────────────────
if logged_in:
    print('✅ 이미 로그인 상태 → 셀4 생략')
else:
    LOGIN_LINK_XPATH  = '//*[@id="wrap"]/div[2]/div[1]/div[1]/ul/li[3]/a'
    IFRAME_XPATH      = '//*[@id="devpopCorpLogin"]'
    LOGIN_BTN_XPATH   = '//*[@id="popCoLoginForm"]/fieldset/div/div/p/button/span'
    
    wait = WebDriverWait(driver, 10)

    # ① 로그인 링크 클릭
    try:
        login_link = wait.until(EC.element_to_be_clickable((By.XPATH, LOGIN_LINK_XPATH)))
        print(f'→ 로그인 링크 클릭: "{login_link.text.strip()}"')
        login_link.click()
        time.sleep(rn())
    except Exception as e:
        print(f'❌ 로그인 링크 없음: {e}')

    # ② iframe 전환
    try:
        iframe = wait.until(EC.presence_of_element_located((By.XPATH, IFRAME_XPATH)))
        driver.switch_to.frame(iframe)
        print('→ iframe 전환 완료')
    except Exception as e:
        print(f'❌ iframe 없음: {e}')

    # # ③ 팝업 내 버튼 클릭
    # try:
    #     btn = wait.until(EC.element_to_be_clickable((By.XPATH, LOGIN_BTN_XPATH)))
    #     print(f'→ 버튼 클릭: "{btn.text.strip()}"')
    #     btn.click()
    #     time.sleep(rn(2.0, 3.5))
    # except Exception as e:
    #     print(f'❌ 버튼 없음: {e}')

    # ④ iframe 해제 후 로그인 결과 확인
    driver.switch_to.default_content()
    time.sleep(rn())

    try:
        els = driver.find_elements(By.XPATH, '//*[contains(text(),"남산HR")]')
        logged_in = bool(els)
    except:
        logged_in = False


→ 로그인 링크 클릭: "로그인"
❌ iframe 없음: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x5bb593+105d3]
	chromedriver!GetHandleVerifier [0x5bb6c4+10704]
	chromedriver!(No symbol) [0x3c1ea0]
	chromedriver!(No symbol) [0x40b28a]
	chromedriver!(No symbol) [0x40b52b]
	chromedriver!(No symbol) [0x44d052]
	chromedriver!(No symbol) [0x42d7c4]
	chromedriver!(No symbol) [0x44a936]
	chromedriver!(No symbol) [0x42d516]
	chromedriver!(No symbol) [0x4008e9]
	chromedriver!(No symbol) [0x4016a4]
	chromedriver!GetHandleVerifier [0x843014+298054]
	chromedriver!GetHandleVerifier [0x83e603+293643]
	chromedriver!GetHandleVerifier [0x85ea05+2b3a45]
	chromedriver!GetHandleVerifier [0x5d54e8+2a528]
	chromedriver!GetHandleVerifier [0x5dcd1d+31d5d]
	chromedriver!GetHandleVerifier [0x5c3e68+18ea8]
	chromedriver!GetHandleVerifier [0x5c4015+19055]
	chromedriver!GetHandleVerifier [0x5ad65f+269f]
	KERNEL32!BaseThreadInitThunk [0x75315d49+19]
	ntdll!RtlInitializeExceptionChain [0x7796d83b+6b]
	ntdll!RtlGetAppContaine

In [67]:
# ── [셀4-2] 로그인 확인 → 인재검색 이동 ────────────────
import time

USER_NAME_XPATH = '//*[@id="wrap"]/div[2]/div[1]/div[1]/ul/li[1]/a'
TALENT_URL      = 'https://www.jobkorea.co.kr/corp/person/find'

wait = WebDriverWait(driver, 10)
time.sleep(rn())

# 남산HR 확인
logged_in = False
try:
    el  = driver.find_element(By.XPATH, USER_NAME_XPATH)
    txt = el.text.strip()
    print(f'   li[1]/a 텍스트: "{txt}"')
    if txt == '남산HR':
        logged_in = True
        print('✅ 로그인 확인 (남산HR)')
except Exception as e:
    print(f'   li[1]/a 없음: {e}')

if not logged_in:
    print('❌ 로그인 실패 — 수동 확인 필요')
else:
    # 인재검색 페이지로 이동
    try:
        driver.get(TALENT_URL)
        time.sleep(rn())
        driver.execute_script('window.stop()')
        print(f'✅ 인재검색 이동 완료 | URL: {driver.current_url}')
    except Exception as e:
        print(f'⚠️  인재검색 이동 실패: {e}')

   li[1]/a 텍스트: "남산HR"
✅ 로그인 확인 (남산HR)
✅ 인재검색 이동 완료 | URL: https://www.jobkorea.co.kr/corp/person/find


In [60]:
# # ── [셀5] 인재검색 페이지 이동 ────────────────────────
# # 현재 페이지 로딩 중단 (무한로딩 방지)
# driver.execute_script('window.stop()')
# time.sleep(0.5)

# wait = WebDriverWait(driver, 10)

# # 헤더 "인재검색" 메뉴 클릭
# try:
#     talent_menu = wait.until(EC.element_to_be_clickable((By.XPATH,
#         '/html/body/div[3]/header/div[1]/div/div/div[2]/div/nav/ul/li[8]/a')))
#     talent_menu.click()
#     print('→ 인재검색 메뉴 클릭')
#     time.sleep(2)
#     # 클릭 후 새 페이지가 또 무한로딩하면 중단
#     driver.execute_script('window.stop()')
#     time.sleep(0.5)
# except TimeoutException:
#     print('⚠️  메뉴 클릭 타임아웃 → URL로 직접 이동')
#     try:
#         driver.get('https://www.jobkorea.co.kr/Recruit/Co_Read/C/personsearch')
#     except TimeoutException:
#         driver.execute_script('window.stop()')
#         print('⚠️  URL 이동도 타임아웃 → 강제 중단 후 진행')
# except Exception as e:
#     print(f'⚠️  메뉴 클릭 실패: {e} → URL로 직접 이동')
#     try:
#         driver.get('https://www.jobkorea.co.kr/Recruit/Co_Read/C/personsearch')
#     except TimeoutException:
#         driver.execute_script('window.stop()')

# # 팝업 닫기
# for sel in ['button[aria-label="닫기"]', '.popup-close', '.btn-close', '.modal-close']:
#     for btn in driver.find_elements(By.CSS_SELECTOR, sel):
#         try:
#             if btn.is_displayed():
#                 btn.click()
#                 time.sleep(0.3)
#         except:
#             pass

# print(f'✅ 페이지: {driver.title}')
# print(f'   URL   : {driver.current_url}')

In [68]:
# 셀 6) 검색 조건 입력
import time
from selenium.common.exceptions import TimeoutException as SeleniumTimeout

wait = WebDriverWait(driver, 10)

def jclick(xpath):
    """XPath 요소를 JS click — 일반 click이 막힐 때 우회"""
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].click();", el)
    except Exception as e:
        print(f'  ⚠️  jclick 실패 [{xpath[-40:]}]: {e}')

def is_visible(xpath):
    try:
        return driver.find_element(By.XPATH, xpath).is_displayed()
    except Exception:
        return False

# ── Step 0: 초기화 (클린 상태에서 시작) ──────────────
# //*[@id="dvbtnReset"]/span
try:
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//*[@id="dvbtnReset"]/span'))).click()
    print('→ 초기화 완료')
    time.sleep(rn(1.5, 2.5))
except Exception:
    print('⚠️  초기화 버튼 없음 — 기존 상태에서 진행')

# ── 나이 (45~61) ─────────────────────────────────────
try:
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//*[@id="dvSideFilter"]/ul/li[2]/strong'))).click()
    print('→ 나이 패널 열기')
    time.sleep(rn())
    s = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="txtAgeStart"]')))
    s.clear(); s.send_keys('45')
    e = driver.find_element(By.XPATH, '//*[@id="txtAgeEnd"]')
    e.clear(); e.send_keys('61')
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//*[@id="btnAgeSearch"]'))).click()
    print('→ 나이 45~61 설정')
    time.sleep(rn())
except SeleniumTimeout:
    print('⚠️  나이 타임아웃')

# ── 구직 상태 (구직준비중 + 구직중) ──────────────────
WORK_STATUS = '//*[@id="dvSideFilter"]/ul/li[3]'
PREPARE_JOB    = '//*[@id="dvJobStatus"]/label[1]'
LOOKING_JOB    = '//*[@id="dvJobStatus"]/label[2]'

try:
    if not is_visible(PREPARE_JOB):
        wait.until(EC.element_to_be_clickable(
            (By.XPATH, WORK_STATUS))).click()
        print('→ 구직 상태 패널 열기')
        time.sleep(rn())
    else:
        print('→ 구직 상태 패널 이미 열려있음')
    
    # 구직준비중 체크 여부 확인 후 필요시 클릭
    checkbox1 = wait.until(EC.element_to_be_clickable((By.XPATH, PREPARE_JOB + '/input')))
    if not checkbox1.is_selected():
        jclick(PREPARE_JOB)
        print('→ 구직준비중 클릭')
    else:
        print('→ 구직준비중 이미 선택됨')
    time.sleep(rn())
    
    # 구직중 체크 여부 확인 후 필요시 클릭
    checkbox2 = wait.until(EC.element_to_be_clickable((By.XPATH, LOOKING_JOB + '/input')))
    if not checkbox2.is_selected():
        jclick(LOOKING_JOB)
        print('→ 구직중 클릭')
    else:
        print('→ 구직중 이미 선택됨')
    time.sleep(rn())
except SeleniumTimeout:
    print('⚠️  구직 상태 타임아웃')

# ── 최근 활동일 (3개월 이내) ──────────────────────────
PANEL_TOGGLE_3M = '//*[@id="dvSideFilter"]/ul/li[6]'
LABEL_3M        = '//*[@id="dvUpdateDt"]/label[5]'

try:
    if not is_visible(LABEL_3M):
        wait.until(EC.element_to_be_clickable(
            (By.XPATH, PANEL_TOGGLE_3M))).click()
        print('→ 최근 활동일 패널 열기')
        time.sleep(rn())

    checkbox3m = None
    try:
        checkbox3m = driver.find_element(By.XPATH, LABEL_3M + '/input')
    except Exception:
        try:
            checkbox3m = driver.find_element(By.XPATH, LABEL_3M + '//input')
        except Exception:
            checkbox3m = None

    if checkbox3m is None or not checkbox3m.is_selected():
        jclick(LABEL_3M)
        print('→ 최근 활동일 3개월이내 클릭')
    else:
        print('→ 최근 활동일 3개월이내 이미 선택됨')

    time.sleep(rn())
except SeleniumTimeout:
    print('⚠️  최근 활동일 타임아웃')
except Exception as e:
    print(f'⚠️  최근 활동일 실패: {e}')

# ── 지역 (서울전지역 + 경기전지역) ───────────────────
REGION_BTN   = '//*[@id="dvKeyword"]/div[1]/div[2]/button[2]'
SEOUL_TAB    = '//*[@id="ulWorkingAreaIn"]/li[1]/label'        # 사용자 확인 XPath
SEOUL_ALL    = '//*[@id="ulWorkingAreaLocalIn"]/li[1]/label/span'
GYEONGGI_TAB = '//*[@id="ulWorkingAreaIn"]/li[2]/label'        # 사용자 확인 XPath
GYEONGGI_ALL = '//*[@id="ulWorkingAreaLocalIn"]/li[1]/label/span'

try:
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, REGION_BTN))).click()
    print('→ 지역 패널 열기')
    time.sleep(rn())

    # 서울 탭 → 서울전지역
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, SEOUL_TAB))).click()
    print('→ 서울 탭 클릭')
    time.sleep(rn(1.2, 2.5))            # 서브패널 DOM 갱신 대기
    jclick(SEOUL_ALL)
    print('→ 서울전지역 클릭')
    time.sleep(rn())

    # 경기 탭 → 경기전지역
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, GYEONGGI_TAB))).click()
    print('→ 경기 탭 클릭')
    time.sleep(rn(1.2, 2.5))            # 서브패널 DOM 갱신 대기
    jclick(GYEONGGI_ALL)
    print('→ 경기전지역 클릭')
    time.sleep(rn())

    wait.until(EC.element_to_be_clickable(
        (By.XPATH, REGION_BTN))).click()
    print('→ 지역 패널 닫기')

except SeleniumTimeout:
    print('⚠️  지역 조건 타임아웃')
except Exception as e:
    print(f'⚠️  지역 조건 실패: {e}')

print('\n✅ 셀6 완료')

⚠️  초기화 버튼 없음 — 기존 상태에서 진행
→ 나이 패널 열기
→ 나이 45~61 설정
→ 구직 상태 패널 열기
⚠️  구직 상태 타임아웃
→ 최근 활동일 패널 열기
→ 최근 활동일 3개월이내 클릭
→ 지역 패널 열기
→ 서울 탭 클릭
→ 서울전지역 클릭
→ 경기 탭 클릭
→ 경기전지역 클릭
→ 지역 패널 닫기

✅ 셀6 완료


In [62]:
# 셀 6.5) 직무/스킬 트리 추출 → job_tree.json 저장
import json, time
from pathlib import Path
from selenium.common.exceptions import StaleElementReferenceException

JOB_BTN  = '//*[@id="dvKeyword"]/div[1]/div[2]/button[1]'
BIG_UL   = '//*[@id="ulJobTypeBigIn"]'
MID_UL   = '//*[@id="ulJobTypeMiddleIn"]'
SUB_UL   = '//*[@id="ulJobTypeKeywordIn"]'

wait_tree = WebDriverWait(driver, 10)

# ── 텍스트·코드 추출 헬퍼 ─────────────────────────────
def li_text(li_el):
    """
    li 요소에서 텍스트 추출.
    Selenium .text 는 비가시 요소에서 빈 문자 반환 → JS textContent 사용.
    label > span > li 순으로 시도.
    """
    for xp in ['.//label/span', './/label', '.']:
        try:
            el = li_el.find_element(By.XPATH, xp)
            t = driver.execute_script("return arguments[0].textContent;", el)
            t = (t or '').strip()
            if t:
                return t
        except Exception:
            pass
    return ''

def li_code(li_el):
    """li 요소 하위 input[value]에서 코드 추출"""
    try:
        return (li_el.find_element(By.XPATH, './/input')
                     .get_attribute('value') or '').strip()
    except Exception:
        return ''

# ── 직무/스킬 패널 열기 ──────────────────────────────
try:
    wait_tree.until(EC.element_to_be_clickable((By.XPATH, JOB_BTN))).click()
    print('→ 직무/스킬 패널 열기')
    time.sleep(rn(1.0, 1.5))
except Exception as e:
    print(f'⚠️  패널 열기 실패: {e}')

# ── DOM 구조 진단 (첫 번째 대분류만) ─────────────────
print('\n=== DOM 구조 진단 ===')
big_lis_dbg = driver.find_elements(By.XPATH, BIG_UL + '/li')
print(f'대분류 li 개수: {len(big_lis_dbg)}')
if big_lis_dbg:
    li0 = big_lis_dbg[0]
    print(f'li.text      : {li0.text!r}')
    print(f'li_text()    : {li_text(li0)!r}')
    print(f'li_code()    : {li_code(li0)!r}')
    print(f'outerHTML    : {li0.get_attribute("outerHTML")[:250]}')
print('====================\n')

# ── 대분류 개수 파악 후 추출 시작 ─────────────────────
n_big = len(big_lis_dbg)
job_tree = {"job_categories": []}

for big_idx in range(1, n_big + 1):
    # 대분류 li 재탐색 (stale 방지)
    big_li = driver.find_element(By.XPATH, f'{BIG_UL}/li[{big_idx}]')
    big_name = li_text(big_li)
    big_code = li_code(big_li)

    # 대분류 클릭 → 중분류 패널 갱신
    driver.execute_script("arguments[0].click();",
        big_li.find_element(By.XPATH, './/label'))
    time.sleep(rn(0.8, 1.5))

    category = {
        "job_category_name":  big_name,
        "job_category_code":  big_code,
        "job_mid_categories": []
    }

    # 중분류 개수
    n_mid = len(driver.find_elements(By.XPATH, MID_UL + '/li'))
    print(f'[{big_idx}/{n_big}] {big_name!r} (code={big_code}) — 중분류 {n_mid}개')

    for mid_idx in range(1, n_mid + 1):
        mid_li    = driver.find_element(By.XPATH, f'{MID_UL}/li[{mid_idx}]')
        mid_name  = li_text(mid_li)
        mid_code  = li_code(mid_li)

        # 중분류 클릭 → 소분류 패널 갱신
        driver.execute_script("arguments[0].click();",
            mid_li.find_element(By.XPATH, './/label'))
        time.sleep(rn(0.4, 0.8))

        mid_cat = {
            "job_mid_category_name": mid_name,
            "job_mid_category_code": mid_code,
            "job_sub_categories":    []
        }

        # 소분류 전체 추출
        sub_lis = driver.find_elements(By.XPATH, SUB_UL + '/li')
        for sub_li in sub_lis:
            try:
                sub_name = li_text(sub_li)
                sub_code = li_code(sub_li)
                if sub_name:
                    mid_cat["job_sub_categories"].append({
                        "job_sub_category_name": sub_name,
                        "job_sub_category_code": sub_code
                    })
            except StaleElementReferenceException:
                pass

        print(f'  {mid_idx:3}. {mid_name!r} (소분류 {len(mid_cat["job_sub_categories"])}개)')
        category["job_mid_categories"].append(mid_cat)

    job_tree["job_categories"].append(category)
    print()

# ── 직무/스킬 패널 닫기 ──────────────────────────────
try:
    wait_tree.until(EC.element_to_be_clickable((By.XPATH, JOB_BTN))).click()
    print('→ 직무/스킬 패널 닫기')
except Exception:
    pass

# ── job_tree.json 저장 ────────────────────────────────
output_path = Path(os.getcwd()) / 'job_tree.json'
output_path.write_text(
    json.dumps(job_tree, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

total_mid = sum(len(c["job_mid_categories"]) for c in job_tree["job_categories"])
total_sub = sum(
    len(m["job_sub_categories"])
    for c in job_tree["job_categories"]
    for m in c["job_mid_categories"]
)
print(f'\n✅ 저장 완료: {output_path}')
print(f'   대분류 {len(job_tree["job_categories"])}개 / 중분류 {total_mid}개 / 소분류 {total_sub}개')

⚠️  패널 열기 실패: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x5bb593+105d3]
	chromedriver!GetHandleVerifier [0x5bb6c4+10704]
	chromedriver!(No symbol) [0x3c1ea0]
	chromedriver!(No symbol) [0x40b28a]
	chromedriver!(No symbol) [0x40b52b]
	chromedriver!(No symbol) [0x44d052]
	chromedriver!(No symbol) [0x42d7c4]
	chromedriver!(No symbol) [0x44a936]
	chromedriver!(No symbol) [0x42d516]
	chromedriver!(No symbol) [0x4008e9]
	chromedriver!(No symbol) [0x4016a4]
	chromedriver!GetHandleVerifier [0x843014+298054]
	chromedriver!GetHandleVerifier [0x83e603+293643]
	chromedriver!GetHandleVerifier [0x85ea05+2b3a45]
	chromedriver!GetHandleVerifier [0x5d54e8+2a528]
	chromedriver!GetHandleVerifier [0x5dcd1d+31d5d]
	chromedriver!GetHandleVerifier [0x5c3e68+18ea8]
	chromedriver!GetHandleVerifier [0x5c4015+19055]
	chromedriver!GetHandleVerifier [0x5ad65f+269f]
	KERNEL32!BaseThreadInitThunk [0x75315d49+19]
	ntdll!RtlInitializeExceptionChain [0x7796d83b+6b]
	ntdll!RtlGetAppContainerNamedObjectPath [

In [66]:
# 셀 6.9) 직무/스킬 주차별 조건 설정 (job_tree.json → 월 1~3주차 자동 분배)
import json, datetime, time
from pathlib import Path
from selenium.common.exceptions import StaleElementReferenceException

JOB_BTN = '//*[@id="dvKeyword"]/div[1]/div[2]/button[1]'
BIG_UL  = '//*[@id="ulJobTypeBigIn"]'
MID_UL  = '//*[@id="ulJobTypeMiddleIn"]'
SUB_UL  = '//*[@id="ulJobTypeKeywordIn"]'

wait_job = WebDriverWait(driver, 10)

# ── job_tree.json 로드 ────────────────────────────────
job_tree_path = Path(os.getcwd()) / 'job_tree.json'
job_tree = json.loads(job_tree_path.read_text(encoding='utf-8'))

# ── 주차 계산 ─────────────────────────────────────────
# 1일~7일 = 1주차 / 8일~14일 = 2주차 / 15일 이후 = 3주차
today    = datetime.date.today()
week_no  = min(((today.day - 1) // 7) + 1, 3)
week_key = f'week_{week_no}'
week_jobs = job_tree['weekly_routines'][week_key]['jobs']

print(f'📅 {today}  →  {week_no}주차 루틴 ({week_key})')
print(f'   직무 {len(week_jobs)}개 선택 예정\n')

# ── 텍스트 매칭 헬퍼 ──────────────────────────────────
def find_li_by_text(ul_xpath, target):
    """ul/li 중 JS textContent가 target과 일치 또는 target으로 시작하는 첫 번째 반환"""
    for li in driver.find_elements(By.XPATH, ul_xpath + '/li'):
        try:
            txt = driver.execute_script(
                "return arguments[0].textContent;", li).strip()
            if txt == target or txt.startswith(target):
                return li
        except StaleElementReferenceException:
            pass
    return None

def jclick_li(li_el):
    """li 내 label JS click"""
    try:
        driver.execute_script("arguments[0].click();",
            li_el.find_element(By.XPATH, './/label'))
    except Exception:
        driver.execute_script("arguments[0].click();", li_el)

# ── 직무/스킬 패널 열기 ───────────────────────────────
try:
    wait_job.until(EC.element_to_be_clickable((By.XPATH, JOB_BTN))).click()
    print('→ 직무/스킬 패널 열기')
    time.sleep(rn(1.0, 1.5))
except Exception as e:
    print(f'⚠️  패널 열기 실패: {e}')

# ── 직무 선택 루프 ────────────────────────────────────
ok, fail = 0, 0
last_big, last_mid = None, None   # 동일 대/중분류 재클릭 방지

for job in week_jobs:
    no  = job['no']
    big = job['job_category']
    mid = job['job_mid']
    sub = job['job_sub']

    try:
        # ① 대분류 — 이전과 다를 때만 클릭
        if big != last_big:
            li = find_li_by_text(BIG_UL, big)
            if not li:
                print(f'  ⚠️  [{no:2}] 대분류 없음: {big}')
                fail += 1
                continue
            jclick_li(li)
            time.sleep(rn(0.6, 1.2))
            last_big = big
            last_mid = None     # 대분류 바뀌면 중분류 캐시 초기화

        # ② 중분류 — 이전과 다를 때만 클릭
        if mid != last_mid:
            li = find_li_by_text(MID_UL, mid)
            if not li:
                print(f'  ⚠️  [{no:2}] 중분류 없음: {big} > {mid}')
                fail += 1
                continue
            jclick_li(li)
            time.sleep(rn(0.4, 0.8))
            last_mid = mid

        # ③ 소분류 클릭 (항상 — "전체" 포함)
        li = find_li_by_text(SUB_UL, sub)
        if not li:
            print(f'  ⚠️  [{no:2}] 소분류 없음: {big} > {mid} > {sub}')
            fail += 1
            continue
        jclick_li(li)
        time.sleep(rn(0.3, 0.6))

        ok += 1
        print(f'  ✅ [{no:2}] {big} > {mid} > {sub}')

    except Exception as e:
        print(f'  ❌ [{no:2}] {type(e).__name__}: {str(e)[:60]}')
        fail += 1

# ── 직무/스킬 패널 닫기 ──────────────────────────────
try:
    wait_job.until(EC.element_to_be_clickable((By.XPATH, JOB_BTN))).click()
    print(f'\n→ 직무/스킬 패널 닫기')
except Exception:
    pass

print(f'\n✅ 완료: 성공 {ok}개 / 실패 {fail}개  ({week_no}주차 / 총 {len(week_jobs)}개)')

KeyError: 'weekly_routines'

In [69]:
# ── [셀7] 이력 로드 & 월간 한도 설정 ─────────────────
import json, datetime
from datetime import date, timedelta

MONTHLY_LIMIT = 250      # ← 이달 발송 한도 (100 또는 250)
HISTORY_FILE  = Path(os.getcwd()) / 'recruit_history.json'

# 이력 파일 로드
if HISTORY_FILE.exists():
    history = json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
    print(f'📂 이력 파일 로드: {len(history)}건')
else:
    history = []
    print('📂 이력 없음 (첫 실행)')

# 3개월 이내 발송 URL 집합 (로컬 기준 중복 방지)
cutoff = date.today() - timedelta(days=90)
recently_sent = {
    h['url'] for h in history
    if h.get('url') and h.get('sent_at')
    and date.fromisoformat(h['sent_at']) > cutoff
}

# 이번 달 잔여 한도 계산
this_month      = date.today().strftime('%Y-%m')
month_sent_cnt  = sum(1 for h in history if h.get('sent_at','').startswith(this_month))
remaining_quota = max(0, MONTHLY_LIMIT - month_sent_cnt)

print(f'3개월 내 발송(로컬): {len(recently_sent)}건')
print(f'이번 달 ({this_month}): {month_sent_cnt}건 / {MONTHLY_LIMIT}건')
print(f'남은 한도: {remaining_quota}건')
if remaining_quota == 0:
    print('⚠️  이번 달 한도 소진')

📂 이력 파일 로드: 216건
3개월 내 발송(로컬): 216건
이번 달 (2026-06): 216건 / 250건
남은 한도: 34건


In [70]:
# ── [셀8] 제안 메시지 로드 (proposal_message.md) ───────
MSG_FILE = Path(os.getcwd()) / 'proposal_message.md'

# 파일이 없으면 기본 템플릿 생성 (이후 VSCode에서 직접 편집)
# if not MSG_FILE.exists():
#     MSG_FILE.write_text("""\
# 안녕하세요, {name}님!
# 안녕하세요, 삼성생명(주) 채용의뢰를 받은 남산HR 이인성 팀장입니다.

# 법인영업(GFC) 포지션을 제안드리위해 연락드립니다.
# 기업 CEO 대상 세무·재무·리스크 컨설팅 전문직이며
# "교육의 삼성" 답게 2개월간의 집중 교육을 통해 전문가를 양성합니다.
# 기업 보험에 대한 시장 기회 증가로 삼성 생명에서는 단체 보험 사업의
# 대대적인 확장을 진행하고 있으며 이에대한 일환으로 다양한 인재를 모시고자 합니다.

# 사업 설명회에 참석하시면 상세한 내용을 확인해보실 수 있습니다.
# 관심 있으시면 편하신 시간에 회신 부탁드립니다.

# 이인성 팀장 | 삼성생명 남산HR
# """, encoding='utf-8')
#     print(f'📝 {MSG_FILE} 생성 — VSCode에서 직접 편집 가능')

PROPOSAL_MESSAGE = MSG_FILE.read_text(encoding='utf-8').strip()
print(f'📝 메시지 로드: {MSG_FILE.name} ({len(PROPOSAL_MESSAGE)}자)')
# print('─' * 40)
# print(PROPOSAL_MESSAGE.format(name='홍길동'))
# print('─' * 40)

📝 메시지 로드: proposal_message.md (309자)


In [71]:
# ── [셀10] 포지션 제안 메인 루프 ───────────────────────
import time, json
from datetime import date
from selenium.common.exceptions import (
    NoSuchWindowException, StaleElementReferenceException, WebDriverException,
)

TARGET_COUNT = 14
MAX_PAGES    = 5

PROPOSAL_BTN_XPATH   = '/html/body/div[1]/div[3]/div/div[1]/div/button[1]'
PROPOSAL_HIST_XPATH  = '/html/body/div[1]/div[3]/div/div[1]/div/div[2]'
CONFIRM_3_XPATH      = '//*[@id="dev-send-seletor"]/div/div/div/a'
POSITION_INPUT_XPATH = '//*[@id="posgtitle"]'
DROPDOWN_XP          = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/table/tbody/tr[2]/td/div[2]/div/span/div/div[2]/div'
POSITION_KEYWORD     = 'WF20260603'
CONTENT_XPATH        = '//*[@id="posg_cntnt"]'
BASE                 = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/table/tbody'
MOBILE_BTN           = f'{BASE}/tr[7]/td/div/div/span[1]/div/div[1]/button'
SUBMIT_XPATH         = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/div/button[2]'
CONFIRM_POPUP        = '//*[@id="modalOfferPopup"]/div[2]/div/div[2]/div/div/button'
NEXT_PAGE_XPATH      = '//*[@id="dvResumeList"]/div[1]/div[2]/div/a[1]'
PHONE = ['010', '4935', '7573']
EMAIL = ('insung.lee', 'samsung.com')

def fill_id(fid, val):
    try:
        el = driver.find_element(By.ID, fid)
        driver.execute_script("arguments[0].value='';", el)
        el.send_keys(val)
    except Exception: pass

def fill_xp(xp, val):
    try:
        el = driver.find_element(By.XPATH, xp)
        try: inp = el.find_element(By.XPATH, './/input')
        except: inp = el
        driver.execute_script("arguments[0].value='';", inp)
        inp.send_keys(val)
    except Exception: pass

def dismiss_alert_if_exists():
    try:
        alert = driver.switch_to.alert
        msg   = alert.text
        alert.accept()
        return msg
    except Exception:
        return None

def get_proposal_btn(skip=0):
    count = 0
    try:
        cells = driver.find_elements(By.CSS_SELECTOR, '.tdPosition')
    except Exception:
        return None
    for cell in cells:
        try:
            for b in cell.find_elements(By.TAG_NAME, 'button'):
                if b.text.strip() == '포지션 제안':
                    if count >= skip:
                        return b
                    count += 1
        except (StaleElementReferenceException, Exception):
            pass
    return None

def get_candidate_info(proposal_btn):
    """
    포지션 제안 버튼 → 부모 행 → .tdProfile / .tdSummary 에서
    이름·나이·이력서타이틀·현직장·거주지역 추출.
    버튼 클릭 전(메인 페이지)에 호출해야 함.
    """
    info = {}
    try:
        row = proposal_btn.find_element(By.XPATH,
            './ancestor::*[contains(@class,"tdPosition")][1]/..')

        # 이름·나이·구직상태 (.tdProfile)
        try:
            profile_text = row.find_element(
                By.CSS_SELECTOR, '.tdProfile').text.strip()
            info['profile'] = profile_text          # 예: "강OO (남, 만 55세)\n구직중"
            # 이름만 파싱: "강OO (남, 만 55세)" → "강OO"
            raw = profile_text.split('\n')[0].split('(')[0].strip()
            info['name'] = raw if raw else ''
        except Exception:
            pass

        # 이력서타이틀·현직장 (.tdSummary)
        try:
            summary_el = row.find_element(By.CSS_SELECTOR, '.tdSummary')
            lines = [l.strip() for l in summary_el.text.strip().split('\n') if l.strip()]
            info['title']   = lines[0] if len(lines) > 0 else ''
            info['company'] = lines[1] if len(lines) > 1 else ''
            info['location'] = ''
            try:
                loc_el = summary_el.find_element(By.CSS_SELECTOR,
                    '.location, .area, [class*="location"], [class*="area"]')
                info['location'] = loc_el.text.strip()
            except Exception:
                pass
        except Exception:
            pass

    except Exception:
        pass
    return info

def safe_close_tab(main_win):
    try:
        if driver.current_window_handle != main_win:
            driver.close()
    except Exception: pass
    try:
        driver.switch_to.window(main_win)
    except Exception: pass

def save_log(record):
    try:
        log_file = Path(os.getcwd()) / 'recruit_history.json'
        logs = json.loads(log_file.read_text(encoding='utf-8')) if log_file.exists() else []
        logs.append(record)
        log_file.write_text(json.dumps(logs, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as e:
        print(f'    ⚠️  로그 저장 실패: {e}')

sent = 0

for page_num in range(1, MAX_PAGES + 1):
    if sent >= TARGET_COUNT:
        break

    print(f'\n══ 페이지 {page_num} ══')
    try: driver.execute_script('window.stop()')
    except Exception: pass
    time.sleep(rp())

    page_skip = 0
    page_sent = 0

    while sent < TARGET_COUNT:
        try:
            proposal_btn = get_proposal_btn(skip=page_skip)
        except Exception:
            time.sleep(rp())
            proposal_btn = get_proposal_btn(skip=page_skip)

        if not proposal_btn:
            print(f'   페이지 {page_num} 버튼 소진 → 다음 페이지')
            break

        print(f'\n  [{sent+1}/{TARGET_COUNT}] 처리 시작 (skip={page_skip})')

        # ── 후보자 정보 & 이름 수집 (메인 페이지) ────────────
        cand_info = get_candidate_info(proposal_btn)
        cand_name = cand_info.get('name', '')
        if cand_info:
            print(f'    👤 {cand_name} | {cand_info.get("title","")}')

        main_win    = driver.window_handles[0]
        before_wins = set(driver.window_handles)
        resume_url  = ''
        step        = '버튼 클릭'

        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", proposal_btn)
            time.sleep(0.5)
            proposal_btn.click()
            time.sleep(rp())
        except StaleElementReferenceException:
            print(f'    ⚠️  버튼 stale → 재탐색')
            page_skip += 1
            proposal_btn = get_proposal_btn(skip=page_skip - 1)
            if proposal_btn:
                try:
                    proposal_btn.click()
                    time.sleep(rp())
                except Exception as e2:
                    print(f'    ⚠️  재시도 실패: {type(e2).__name__}')
                    continue
            else:
                continue
        except Exception as e:
            print(f'    ⚠️  {step} 실패: {type(e).__name__}')
            page_skip += 1
            continue

        page_skip += 1

        deadline = time.time() + 3.0
        new_wins = set()
        while time.time() < deadline:
            try:
                new_wins = set(driver.window_handles) - before_wins
            except Exception:
                break
            if new_wins:
                break
            time.sleep(0.3)

        if not new_wins:
            print('    ⚠️  새 탭 없음 — 스킵')
            continue

        try:
            driver.switch_to.window(list(new_wins)[0])
        except NoSuchWindowException:
            print('    ⚠️  새 탭 즉시 닫힘 — 스킵')
            safe_close_tab(main_win)
            continue
        except Exception as e:
            print(f'    ⚠️  탭 전환 실패: {type(e).__name__}')
            safe_close_tab(main_win)
            continue

        try:
            resume_url = driver.current_url
        except Exception:
            resume_url = '(unknown)'

        alert_msg = dismiss_alert_if_exists()
        if alert_msg:
            print(f'    ⚠️  alert → 스킵: "{alert_msg[:50]}"')
            safe_close_tab(main_win)
            time.sleep(rp())
            continue

        try:
            driver.execute_script('window.stop()')
        except NoSuchWindowException:
            print('    ⚠️  탭 닫힘 (stop) — 스킵')
            safe_close_tab(main_win)
            continue
        except Exception:
            pass

        time.sleep(rp())

        try:
            step = '히스토리 확인'
            hist_text = ''
            try:
                hist_text = driver.find_element(By.XPATH, PROPOSAL_HIST_XPATH).text.strip()
            except Exception:
                pass

            if hist_text and '제안 내역이 없습니다' not in hist_text:
                print('    ⏭️  이미 제안됨 — 스킵')
                raise StopIteration

            step = '제안 버튼 클릭'
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, PROPOSAL_BTN_XPATH))).click()
            time.sleep(rp())

            step = '건수 차감 확인'
            try:
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, CONFIRM_3_XPATH))).click()
                time.sleep(rp())
            except Exception:
                pass

            step = '포지션 선택'
            inp_pos = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, POSITION_INPUT_XPATH)))
            inp_pos.clear()
            inp_pos.send_keys(POSITION_KEYWORD)
            time.sleep(rp())
            radio = driver.find_element(By.XPATH,
                f'{DROPDOWN_XP}//li[contains(.,"{POSITION_KEYWORD}")]/input')
            driver.execute_script("arguments[0].click();", radio)
            time.sleep(rp())

            step = '내용 입력'
            ta = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, CONTENT_XPATH)))
            ta.clear()
            driver.execute_script("arguments[0].value='';", ta)
            # ── {name} 치환 후 발송 ────────────────────────
            msg = PROPOSAL_MESSAGE.replace('{name}', cand_name) if cand_name else PROPOSAL_MESSAGE
            if not cand_name:
                print('    ⚠️  이름 추출 실패 — {name} 미치환 상태로 발송')
            ta.send_keys(msg)

            step = '담당자 정보'
            fill_id('lb_Ofc_Man_Name', '이인성')
            fill_id('lb_dept_name',    '남산HR')
            fill_xp(f'{BASE}/tr[6]/td/div/div/span[1]', PHONE[0])
            fill_xp(f'{BASE}/tr[6]/td/div/div/span[2]', PHONE[1])
            fill_xp(f'{BASE}/tr[6]/td/div/div/span[3]', PHONE[2])
            driver.find_element(By.XPATH, MOBILE_BTN).click()
            time.sleep(rp())
            items = driver.find_elements(By.XPATH,
                f'{MOBILE_BTN}/following::ul[1]/li | '
                f'{MOBILE_BTN}/following::div[contains(@class,"list")][1]//li')
            if len(items) >= 2:
                try:
                    driver.execute_script("arguments[0].click();",
                        items[1].find_element(By.XPATH, './/input'))
                except Exception:
                    driver.execute_script("arguments[0].click();", items[1])
            time.sleep(rp())
            fill_id('GuinOfcMan_Entity_Mobile_No2', PHONE[1])
            fill_id('GuinOfcMan_Entity_Mobile_No3', PHONE[2])
            fill_id('GuinOfcMan_Entity_Email1', EMAIL[0])
            fill_id('GuinOfcMan_Entity_Email2', EMAIL[1])

            step = '제안 보내기'
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, SUBMIT_XPATH))).click()
            time.sleep(rp())

            step = '완료 팝업'
            try:
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, CONFIRM_POPUP))).click()
                time.sleep(rp())
            except Exception:
                pass

            sent      += 1
            page_sent += 1
            save_log({
                'sent_at' : date.today().isoformat(),
                'url'     : resume_url,
                'page'    : page_num,
                'idx'     : page_skip - 1,
                'name'    : cand_name,
                'profile' : cand_info.get('profile',  ''),
                'title'   : cand_info.get('title',    ''),
                'company' : cand_info.get('company',  ''),
                'location': cand_info.get('location', ''),
            })
            print(f'    ✅ 발송 완료 ({sent}/{TARGET_COUNT}) → {cand_name} | {resume_url}')

        except StopIteration:
            pass
        except NoSuchWindowException:
            print(f'    ❌ 탭 닫힘 [{step}] — 스킵')
        except Exception as e:
            etype = type(e).__name__
            emsg  = str(e).split('\n')[0]
            print(f'    ❌ 오류 [{step}] {etype}: {emsg[:100]}')
        finally:
            safe_close_tab(main_win)

        w = rp(3.0, 8.0)
        print(f'    → {w:.1f}초 대기')
        time.sleep(w)

    print(f'   페이지 {page_num} 소계: {page_sent}건 발송')

    if sent < TARGET_COUNT:
        try:
            nxt = driver.find_element(By.XPATH, NEXT_PAGE_XPATH)
            print(f'   다음 페이지 버튼: "{nxt.text.strip()}"')
            nxt.click()
            time.sleep(rp(2.0, 4.0))
            print('→ 다음 페이지 이동')
        except Exception:
            print('마지막 페이지 — 종료')
            break

print(f'\n🏁 완료: {sent}/{TARGET_COUNT}건 발송')

try:
    log_file = Path(os.getcwd()) / 'recruit_history.json'
    if log_file.exists():
        logs      = json.loads(log_file.read_text(encoding='utf-8'))
        today     = date.today().isoformat()
        today_cnt = sum(1 for l in logs if l.get('sent_at') == today)
        print(f'📋 로컬 로그: 오늘 {today_cnt}건 / 누적 {len(logs)}건 ({log_file})')
except Exception:
    pass


══ 페이지 1 ══

  [1/14] 처리 시작 (skip=0)
    👤 정OO | 경력 31년6개월
    ❌ 오류 [제안 버튼 클릭] TimeoutException: Message: 
    → 7.7초 대기

  [1/14] 처리 시작 (skip=1)
    👤 박OO | 경력 13년
    ❌ 오류 [제안 버튼 클릭] TimeoutException: Message: 
    → 5.7초 대기

  [1/14] 처리 시작 (skip=2)
    👤 김OO | 경력 26년4개월
    ⏭️  이미 제안됨 — 스킵
    → 7.0초 대기

  [1/14] 처리 시작 (skip=3)
    👤 김OO | 경력 17년1개월
    ❌ 오류 [제안 버튼 클릭] TimeoutException: Message: 
    → 4.2초 대기

  [1/14] 처리 시작 (skip=4)
    👤 김OO | 경력 6년
    ✅ 발송 완료 (1/14) → 김OO | https://www.jobkorea.co.kr/corp/person/find/resume/view?rNo=30553276
    → 3.1초 대기

  [2/14] 처리 시작 (skip=5)
    👤 김OO | 경력 28년9개월
    ❌ 오류 [제안 버튼 클릭] TimeoutException: Message: 
    → 7.8초 대기

  [2/14] 처리 시작 (skip=6)
    👤 정OO | 경력 18년6개월
    ✅ 발송 완료 (2/14) → 정OO | https://www.jobkorea.co.kr/corp/person/find/resume/view?rNo=125941
    → 6.5초 대기

  [3/14] 처리 시작 (skip=7)
    👤 천OO | 추천
    ❌ 오류 [제안 버튼 클릭] TimeoutException: Message: 
    → 4.6초 대기

  [3/14] 처리 시작 (skip=8)
    👤 원OO | 추천
    ✅ 발송 완료 (3/14) → 원O

In [ ]:
# 팝업창에서 확인 버튼 누르기를 아래 xpath로 시도해야 해.
# //*[@id="dev-send-seletor"]/div/div/div/a

In [ ]:
# ── [셀9-진단] 검색결과 페이지 셀렉터 파악 ─────────────
# 셀9 실행 전에 이 셀을 먼저 실행해서 실제 후보자 항목 셀렉터를 확인하세요.

# print(f'현재 URL: {driver.current_url}')
# print(f'페이지 제목: {driver.title}')
# print()

# # 후보자가 들어있을 것 같은 셀렉터 후보를 모두 시도
# candidates_to_check = [
#     '.resumeBox', '.person-list-item', '.list-item', 'li.item',
#     '.devPersonList li', '.resultInfo', '#listPerson li',
#     '.listType02 li', '.co_person li', '.list_person li',
#     'ul.list li', 'div.list li', '.srchPerson li',
#     '.personList li', 'li.person', '.resumeList li',
#     '.srchList li', 'tr.person', '.tbl_list tr',
# ]

# found = []
# for sel in candidates_to_check:
#     els = driver.find_elements(By.CSS_SELECTOR, sel)
#     if els:
#         found.append((sel, len(els)))
#         print(f'✅ {sel:<35} → {len(els)}개 발견')

# if not found:
#     print('❌ 후보 셀렉터 없음 — 페이지 소스에서 직접 탐색합니다.')
#     print()
#     # 이름처럼 보이는 텍스트가 있는 요소 탐색
#     src = driver.page_source
#     # 후보자 이름 패턴 (OO님 등)
#     import re
#     # li 또는 div 태그 클래스명 추출
#     classes = re.findall(r'class="([^"]+)"', src)
#     class_counts = {}
#     for c in classes:
#         for cls in c.split():
#             class_counts[cls] = class_counts.get(cls, 0) + 1
#     # 10번 이상 반복되는 클래스 (목록 항목일 가능성)
#     repeated = sorted([(v, k) for k, v in class_counts.items() if v >= 10], reverse=True)
#     print('10회 이상 반복 클래스 (후보 셀렉터):')
#     for cnt, cls in repeated[:20]:
#         print(f'  .{cls:<40} ({cnt}회)')

현재 URL: https://www.jobkorea.co.kr/corp/person/find
페이지 제목: 인재검색 - 직무 경험이 풍부한 우수 인재 | 잡코리아

❌ 후보 셀렉터 없음 — 페이지 소스에서 직접 탐색합니다.

10회 이상 반복 클래스 (후보 셀렉터):
  .js-skillSearch                           (617회)
  .js-kwrdSearch                            (444회)
  .keywordBox                               (268회)
  .dvResumeLink                             (200회)
  .careerIcon                               (200회)
  .userInfoBox                              (137회)
  .userInfo                                 (137회)
  .devButtonScrap                           (137회)
  .career                                   (137회)
  .keywordSkill                             (136회)
  .keywordJob                               (130회)
  .hashTagBox                               (130회)
  .title                                    (117회)
  .inner                                    (106회)
  .btnClose                                 (106회)
  .detail                                   (101회)
  .tdSummary                      

In [ ]:
# ── [셀30] 브라우저 종료 (선택) ───────────────────────
# 세션 유지를 원하면 이 셀 실행 X
# 완전히 끝냈을 때만 실행

# driver.quit()
# print('✅ 브라우저 종료')

print('브라우저 유지 중 (다음 실행 시 로그인 불필요)')
print('종료하려면 driver.quit() 주석 해제 후 실행')